In [1]:
# Cell 1: Imports & Setup

import sys
import os
sys.path.append(os.path.abspath('..')) 

import pandas as pd
import plotly.graph_objects as go
from src.features import FeatureEngineer
from src.regime import RegimeDetector

# 1. Load Data
df = pd.read_parquet('../data/raw/SPY.parquet')

# 2. CRITICAL FIX: Remove Duplicate Indices
# This fixes "ValueError: cannot reindex on an axis with duplicate labels"
df = df[~df.index.duplicated(keep='first')]

# 3. Engineer Features
fe = FeatureEngineer(df)
fe.add_volatility_features().add_trend_features().add_volume_features()
data = fe.get_features()

# 4. Get Aligned Prices
# We retrieve 'Close' directly from the FeatureEngineer's internal dataframe
# because we know it has already flattened the columns correctly.
prices = fe.df.loc[data.index, 'Close']

print(f"Data Loaded: {len(data)} rows.")
print("Feature Engineering Complete.")

Data Loaded: 2204 rows.
Feature Engineering Complete.


In [2]:
# Cell 2: Train Model

# Initialize Detector
# Note: We must update the feature_cols to match the new Capitalized names
regime_engine = RegimeDetector(n_components=4)
regime_engine.feature_cols = ['Vol_ratio', 'Momentum', 'Vol_short']

# Train
regime_engine.fit(data)

# Predict
data['regime'] = regime_engine.predict(data)

# Print Statistics to identify regimes
print("Regime Statistics (Mean values):")
print(data.groupby('regime')[['Momentum', 'Vol_short', 'Vol_ratio']].mean())

Model trained. Converged: True
Regime Statistics (Mean values):
        Momentum  Vol_short  Vol_ratio
regime                                
0       0.044759   0.093387   0.868432
1      -0.148372   0.591284   1.061178
2       0.075293   0.183375   0.756134
3      -0.018545   0.211545   1.193074


In [3]:
# Cell 3: Visualization

# Create a colored chart
fig = go.Figure()

# Define colors for regimes (adjust based on the stats printed above)
# We will just map 0,1,2,3 to distinct colors for now
colors = ['green', 'red', 'blue', 'orange']

for regime in range(4):
    subset = data[data['regime'] == regime]
    # We plot markers to show where this regime is active
    fig.add_trace(go.Scatter(
        x=subset.index, 
        y=prices.loc[subset.index],
        mode='markers',
        marker=dict(size=3, color=colors[regime]),
        name=f'Regime {regime}'
    ))

fig.update_layout(title="Market Regimes on SPY", template="plotly_dark")
fig.show()